
# Multisubject network-motif simulation using the paper's **actual MS-AA code**

This notebook is designed to run in the **same project environment** as:

- `001_fit_msaa_flexible.ipynb`
- `002_decode_msaa.ipynb`

Unlike the earlier simulation notebook, this version uses the same:

- `Subject` class,
- `multi_subject_aa(...)`,
- `fit_msaa(...)` wrapper structure,
- spatial MS-AA orientation,
- `sXC @ S` reconstruction,
- node-wise z-scoring of reconstructed data,

that are used in the empirical manuscript analyses.

The only thing being replaced is the input data: instead of the Pieman HTFA matrices, we generate multisubject synthetic data with known networks, subnetworks, and latent distributed motifs.

## Main question

> If observations are generated as mixtures of overlapping, nonorthogonal extremal network motifs, does MS-AA recover the true motifs more directly than matched-dimensional PCA?

This notebook deliberately separates three questions:

1. **Reconstruction** — how well does each model reconstruct the observed data?
2. **Representation** — can the fitted model represent the known true motifs?
3. **Disentanglement / interpretability** — do individual recovered dimensions correspond one-to-one with the known motifs?

The third question is the one for which AA has the clearest theoretical advantage. PCA may still reconstruct very well.

A final negative-control simulation generates data from orthogonal Gaussian latent factors, where PCA should be favored.


In [ ]:

# ============================================================
# Configuration
# ============================================================

ANALYSIS_TYPE = "spatial"   # this simulation focuses on spatial motifs
RNG_SEED = 42

HELPERS_DIR = "helpers"

# Same MS-AA options used in 001_fit_msaa_flexible.ipynb
MSAA_OPTS = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)

# ---------------- Simulation size ----------------
# Kept moderate for an initial run.
N_SUBJECTS = 16
T_PER_SUBJECT = 150

N_NETWORKS = 6
SUBNETWORKS_PER_NETWORK = 2
NODES_PER_SUBNETWORK = 10

N_TRUE_MOTIFS = 6
K_FIT = N_TRUE_MOTIFS

# ---------------- Generative parameters ----------------
# Sparse Dirichlet weights put many observations near convex-hull extremes.
DIRICHLET_ALPHA = 0.10

# Between-subject spatial variability
SUBJECT_MULTIPLICATIVE_SD = 0.08
SUBJECT_ADDITIVE_SD = 0.04

# Measurement noise
NOISE_SD = 0.25


In [ ]:

# ============================================================
# Imports -- same MS-AA implementation as the empirical notebook
# ============================================================

%matplotlib inline

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment, minimize
from sklearn.decomposition import PCA

sys.path.append(HELPERS_DIR)

# IMPORTANT: this is the same implementation imported by
# 001_fit_msaa_flexible.ipynb
from MultiSubject_AA import Subject, multi_subject_aa

print("Imported Subject and multi_subject_aa from:", HELPERS_DIR)



## 1. The exact MS-AA fitting/reconstruction logic used in the paper

The functions below are copied/adapted directly from the current fitting and decoding notebooks.

For spatial AA:

\[
X_s \approx sXC_s S_s,
\]

where `sXC` has shape \(T \times K\) and `S` has shape \(K \times V\).

The recovered spatial archetypes are therefore the rows of `S`.


In [ ]:

def to_float_array(x):
    return np.array(x, dtype=float)


def coerce_TxV(x):
    """
    Simulation matrices are explicitly generated as T x V,
    so preserve that orientation.
    """
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(
            f"Expected a 2D T x V matrix, got {x.shape}"
        )

    # Match the empirical preprocessing step:
    # center each node across time.
    x = x - x.mean(axis=0, keepdims=True)

    return x.astype(np.float32, copy=False)


def orient_subject_matrix(X_raw, analysis_type):
    Xtv = coerce_TxV(X_raw)
    return Xtv if analysis_type == "spatial" else Xtv.T


def build_subject_objects(subject_list, analysis_type):
    return [
        Subject(
            X=orient_subject_matrix(x, analysis_type),
            sX=orient_subject_matrix(x, analysis_type),
        )
        for x in subject_list
    ]


def expected_shape_string(example_X_raw, analysis_type, K):
    Xtv = coerce_TxV(example_X_raw)
    T, V = Xtv.shape
    return (
        f"sXC ~ ({T}, {K}), S ~ ({K}, {V})"
        if analysis_type == "spatial"
        else f"sXC ~ ({V}, {K}), S ~ ({K}, {T})"
    )


def fit_msaa(subject_list, K, analysis_type, opts=None, verbose=True):
    # Same wrapper structure as 001_fit_msaa_flexible.ipynb
    opts = dict(MSAA_OPTS if opts is None else opts)

    subj_objs = build_subject_objects(
        subject_list,
        analysis_type,
    )

    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(
        subj_objs,
        noc=K,
        opts=opts,
    )

    if verbose:
        print(
            f"[{analysis_type.upper()} AA] "
            f"K={K} | VarExpl={varexpl*100:.2f}%"
        )
        print(
            "subj0 sXC shape:",
            np.asarray(results_subj[0]["sXC"]).shape,
        )
        print(
            "subj0 S shape:",
            np.asarray(results_subj[0]["S"]).shape,
        )
        print(
            "Expected:",
            expected_shape_string(
                subject_list[0],
                analysis_type,
                K,
            ),
        )

    return {
        "results_subj": results_subj,
        "C": C,
        "cost_fun": cost_fun,
        "varexpl": varexpl,
        "secs": secs,
        "K": K,
    }


def reconstruct_from_archetypes(
    sub,
    analysis_type,
    archetype_indices=None,
    zscore=True,
):
    # Same reconstruction logic as 002_decode_msaa.ipynb
    sXC = np.asarray(sub["sXC"], dtype=float)
    S = np.asarray(sub["S"], dtype=float)

    if archetype_indices is None:
        idx = np.arange(S.shape[0])
    else:
        idx = np.asarray(
            list(archetype_indices),
            dtype=int,
        )

    if analysis_type == "spatial":
        Xhat = sXC[:, idx] @ S[idx, :]
    elif analysis_type == "temporal":
        Xhat = (sXC[:, idx] @ S[idx, :]).T
    else:
        raise ValueError(
            "analysis_type must be 'spatial' or 'temporal'"
        )

    if zscore:
        Xhat = (
            Xhat - Xhat.mean(axis=0, keepdims=True)
        ) / (
            Xhat.std(axis=0, keepdims=True) + 1e-8
        )

    return Xhat



## 2. Construct known networks, subnetworks, and overlapping motifs

There are six large-scale networks. Each contains two subnetworks. The six true latent motifs are deliberately:

- distributed across multiple large-scale networks,
- overlapping,
- nonorthogonal,
- differentiated partly by which **subnetwork** is active.

This is intentionally different from a trivial simulation in which each latent factor equals one isolated network.


In [ ]:

def make_true_motifs():
    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    n_nodes = (
        n_subnetworks
        * NODES_PER_SUBNETWORK
    )

    # Each motif recruits three subnetworks.
    # Related motifs deliberately overlap while differing
    # in one or more subnetwork identities.
    motif_subnetworks = [
        [0,  3,  8],
        [1,  3,  9],
        [2,  5, 10],
        [2,  4, 11],
        [0,  6,  7],
        [1,  6,  8],
    ]

    motifs = np.zeros(
        (len(motif_subnetworks), n_nodes),
        dtype=float,
    )

    # Small shared background
    motifs[:] = 0.05

    for k, active in enumerate(motif_subnetworks):
        for subnet in active:
            lo = subnet * NODES_PER_SUBNETWORK
            hi = lo + NODES_PER_SUBNETWORK

            # Strong subnetwork activation
            motifs[k, lo:hi] = 1.0

    return motifs, motif_subnetworks


TRUE_MOTIFS_RAW, TRUE_SUBNETWORKS = make_true_motifs()

print("True motifs:", TRUE_MOTIFS_RAW.shape)
print(
    "Total subnetworks:",
    N_NETWORKS * SUBNETWORKS_PER_NETWORK,
)
print("Total nodes:", TRUE_MOTIFS_RAW.shape[1])


In [ ]:

def plot_pattern_matrix(
    patterns,
    title,
    ylabel="Pattern",
):
    fig, ax = plt.subplots(figsize=(12, 4))

    im = ax.imshow(
        patterns,
        aspect="auto",
    )

    n_nodes = patterns.shape[1]

    # subnetwork boundaries
    for x in range(
        NODES_PER_SUBNETWORK,
        n_nodes,
        NODES_PER_SUBNETWORK,
    ):
        ax.axvline(
            x - 0.5,
            linewidth=0.4,
            alpha=0.35,
        )

    # network boundaries
    network_width = (
        SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    for x in range(
        network_width,
        n_nodes,
        network_width,
    ):
        ax.axvline(
            x - 0.5,
            linewidth=1.2,
        )

    ax.set_xlabel(
        "Nodes grouped by network → subnetwork"
    )
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    fig.colorbar(
        im,
        ax=ax,
        shrink=0.8,
    )

    plt.show()


plot_pattern_matrix(
    TRUE_MOTIFS_RAW,
    "Raw ground-truth distributed motifs",
    ylabel="True motif",
)



## 3. Generate multisubject data from convex mixtures of the true motifs

For subject \(s\), we create a slightly perturbed version of each group motif. At every time point,

\[
X_s(t) =
\sum_k w_{s,t,k} M_{s,k}
+ \epsilon_{s,t},
\]

with

\[
w_{s,t,k}\ge 0,\qquad
\sum_k w_{s,t,k}=1.
\]

A sparse Dirichlet distribution makes some observations lie close to the latent extremes while still generating mixtures throughout the convex hull.

We then apply the same basic across-subject feature standardization used by the empirical pipeline. The true motifs are transformed through the **same affine normalization** so that ground-truth comparisons remain meaningful.


In [ ]:

def simulate_multisubject_data(
    true_motifs,
    seed=RNG_SEED,
    noise_sd=NOISE_SD,
    dirichlet_alpha=DIRICHLET_ALPHA,
):
    rng = np.random.default_rng(seed)

    K, V = true_motifs.shape

    raw_subjects = []
    true_weights = []
    subject_specific_motifs = []

    for s in range(N_SUBJECTS):

        multiplicative = (
            1.0
            + rng.normal(
                0,
                SUBJECT_MULTIPLICATIVE_SD,
                size=(K, V),
            )
        )

        additive = rng.normal(
            0,
            SUBJECT_ADDITIVE_SD,
            size=(K, V),
        )

        subj_motifs = (
            true_motifs * multiplicative
            + additive
        )

        W = rng.dirichlet(
            np.full(K, dirichlet_alpha),
            size=T_PER_SUBJECT,
        )

        X = W @ subj_motifs

        X += rng.normal(
            0,
            noise_sd,
            size=X.shape,
        )

        raw_subjects.append(X)
        true_weights.append(W)
        subject_specific_motifs.append(
            subj_motifs
        )

    return {
        "raw_subjects": raw_subjects,
        "true_weights": true_weights,
        "subject_specific_motifs": subject_specific_motifs,
    }


def normalize_across_subjects(subject_list):
    """
    Matches the normalize_across_fallback logic used in
    001_fit_msaa_flexible.ipynb, while also returning mu/sd
    so the true motifs can be transformed identically.
    """
    subjects = [
        np.asarray(x, dtype=float)
        for x in subject_list
    ]

    stacked = np.vstack(subjects)

    mu = stacked.mean(
        axis=0,
        keepdims=True,
    )

    sd = (
        stacked.std(
            axis=0,
            keepdims=True,
        )
        + 1e-8
    )

    normalized = [
        ((x - mu) / sd).astype(
            np.float32,
            copy=False,
        )
        for x in subjects
    ]

    return normalized, mu, sd


sim = simulate_multisubject_data(
    TRUE_MOTIFS_RAW
)

subject_data, SIM_MU, SIM_SD = (
    normalize_across_subjects(
        sim["raw_subjects"]
    )
)

# Transform group-level true motifs by the exact same
# affine normalization.
TRUE_MOTIFS = (
    TRUE_MOTIFS_RAW - SIM_MU
) / SIM_SD

print(
    "Subjects:",
    len(subject_data),
)

print(
    "Subject matrix:",
    subject_data[0].shape,
)

plot_pattern_matrix(
    TRUE_MOTIFS,
    "Ground-truth motifs after the same feature normalization",
    ylabel="True motif",
)



## 4. Fit the **actual multisubject AA model**

This is the same `fit_msaa(...)` path as the empirical analysis. No separate AA implementation is used here.


In [ ]:

msaa_fit = fit_msaa(
    subject_data,
    K=K_FIT,
    analysis_type=ANALYSIS_TYPE,
    opts=MSAA_OPTS,
    verbose=True,
)

print("\nReturned fit keys:")
print(msaa_fit.keys())

print("\nC type/shape diagnostic:")
try:
    print(type(msaa_fit["C"]))
    print(np.asarray(msaa_fit["C"]).shape)
except Exception as e:
    print("Could not coerce C directly:", e)



### Group representation of the recovered spatial archetypes

`multi_subject_aa` returns subject-specific `S` matrices. We first inspect each subject's recovered spatial archetypes and compute a group-average archetype matrix.

Because the same multisubject fit jointly estimates the subjects, archetype indices are expected to be aligned across subjects within the fit. The cell below also reports their cross-subject consistency rather than assuming it.


In [ ]:

S_subjects = np.stack(
    [
        np.asarray(sub["S"], dtype=float)
        for sub in msaa_fit["results_subj"]
    ],
    axis=0,
)

# subjects x K x V
print("S_subjects:", S_subjects.shape)

S_GROUP = S_subjects.mean(axis=0)

# Cross-subject consistency of each archetype
arch_consistency = []

for k in range(K_FIT):
    vals = []

    for i in range(N_SUBJECTS):
        for j in range(i + 1, N_SUBJECTS):
            r = np.corrcoef(
                S_subjects[i, k],
                S_subjects[j, k],
            )[0, 1]
            vals.append(r)

    arch_consistency.append(
        np.nanmean(vals)
    )

consistency_df = pd.DataFrame({
    "archetype": np.arange(K_FIT),
    "mean_between_subject_r": arch_consistency,
})

display(consistency_df)

plot_pattern_matrix(
    S_GROUP,
    "Recovered group-average MS-AA spatial archetypes",
    ylabel="Recovered archetype",
)



## 5. Fit matched-dimensional PCA to exactly the same normalized data

PCA is fit to all simulated observations concatenated across subjects, using the same \(K\) as MS-AA.

This comparison intentionally allows PCA to do what it does best: find variance-maximizing orthogonal directions.


In [ ]:

X_STACK = np.vstack(subject_data)

pca = PCA(
    n_components=K_FIT,
    random_state=RNG_SEED,
)

PCA_SCORES = pca.fit_transform(
    X_STACK
)

XHAT_PCA_STACK = pca.inverse_transform(
    PCA_SCORES
)

print(
    "PCA cumulative explained variance:",
    pca.explained_variance_ratio_.sum(),
)



## 6. Data reconstruction comparison

PCA is optimal for ordinary least-squares low-rank approximation, so it would not be surprising if PCA reconstructs better. This is included as a fidelity check, **not** as the primary AA advantage.


In [ ]:

# MS-AA reconstructions WITHOUT the decoder's final z-score,
# because MSE should be evaluated in the original normalized scale.
XHAT_AA_SUBJECTS = [
    reconstruct_from_archetypes(
        sub,
        ANALYSIS_TYPE,
        zscore=False,
    )
    for sub in msaa_fit["results_subj"]
]

XHAT_AA_STACK = np.vstack(
    XHAT_AA_SUBJECTS
)

aa_mse = np.mean(
    (X_STACK - XHAT_AA_STACK) ** 2
)

pca_mse = np.mean(
    (X_STACK - XHAT_PCA_STACK) ** 2
)

reconstruction_df = pd.DataFrame({
    "method": ["MS-AA", "PCA"],
    "MSE": [aa_mse, pca_mse],
})

display(reconstruction_df)



## 7. One-to-one recovery of the known motifs

This is the key disentanglement test.

For each method, we compute a pattern-correlation matrix between the known true motifs and the individually recovered dimensions and use the Hungarian algorithm to find the optimal one-to-one matching.

- **MS-AA:** true motifs are matched directly to recovered archetypes.
- **PCA:** true motifs are matched to PCA loading vectors; absolute correlation is used because PCA signs are arbitrary.

This does **not** ask whether the PCA *subspace* contains the motifs. It asks whether its individual dimensions correspond to the underlying motifs.


In [ ]:

def pattern_corr_matrix(
    A,
    B,
    absolute=False,
):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    A0 = A - A.mean(
        axis=1,
        keepdims=True,
    )

    B0 = B - B.mean(
        axis=1,
        keepdims=True,
    )

    A0 /= (
        np.linalg.norm(
            A0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    B0 /= (
        np.linalg.norm(
            B0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    C = A0 @ B0.T

    if absolute:
        C = np.abs(C)

    return C


def optimal_match(
    true_patterns,
    recovered_patterns,
    absolute=False,
):
    C = pattern_corr_matrix(
        true_patterns,
        recovered_patterns,
        absolute=absolute,
    )

    true_idx, recovered_idx = (
        linear_sum_assignment(-C)
    )

    matched = C[
        true_idx,
        recovered_idx,
    ]

    return {
        "similarity": C,
        "true_idx": true_idx,
        "recovered_idx": recovered_idx,
        "matched": matched,
    }


aa_match = optimal_match(
    TRUE_MOTIFS,
    S_GROUP,
    absolute=False,
)

pca_match = optimal_match(
    TRUE_MOTIFS,
    pca.components_,
    absolute=True,
)

motif_recovery_df = pd.DataFrame({
    "method": ["MS-AA", "PCA"],
    "mean_matched_r": [
        np.mean(aa_match["matched"]),
        np.mean(pca_match["matched"]),
    ],
    "median_matched_r": [
        np.median(aa_match["matched"]),
        np.median(pca_match["matched"]),
    ],
    "min_matched_r": [
        np.min(aa_match["matched"]),
        np.min(pca_match["matched"]),
    ],
})

display(motif_recovery_df)


In [ ]:

def plot_similarity_matrix(
    C,
    title,
):
    fig, ax = plt.subplots(
        figsize=(5.5, 4.5)
    )

    im = ax.imshow(
        C,
        vmin=0,
        vmax=1,
        aspect="auto",
    )

    ax.set_xlabel("Recovered dimension")
    ax.set_ylabel("True motif")
    ax.set_title(title)

    fig.colorbar(
        im,
        ax=ax,
        label="Pattern similarity",
    )

    plt.show()


plot_similarity_matrix(
    aa_match["similarity"],
    "True motif ↔ MS-AA archetype similarity",
)

plot_similarity_matrix(
    pca_match["similarity"],
    "True motif ↔ PCA component similarity",
)


In [ ]:

# Put both recovered representations in true-motif order
AA_ORDERED = S_GROUP[
    aa_match["recovered_idx"]
].copy()

PCA_ORDERED = pca.components_[
    pca_match["recovered_idx"]
].copy()

# Flip PCA signs only for visualization
for i in range(K_FIT):
    r = np.corrcoef(
        TRUE_MOTIFS[i],
        PCA_ORDERED[i],
    )[0, 1]

    if r < 0:
        PCA_ORDERED[i] *= -1


plot_pattern_matrix(
    TRUE_MOTIFS,
    "Ground truth",
    ylabel="True motif",
)

plot_pattern_matrix(
    AA_ORDERED,
    "MS-AA archetypes optimally matched to ground truth",
    ylabel="Matched archetype",
)

plot_pattern_matrix(
    PCA_ORDERED,
    "PCA components optimally matched to ground truth",
    ylabel="Matched component",
)



## 8. Subnetwork-level recovery

Node-level correlations may be stricter than the interpretation we ultimately care about. Here each pattern is collapsed to its 12 known subnetworks, and the same one-to-one recovery test is repeated.

If AA is truly preserving the identity of distributed motifs, it should recover not only node-level patterns but the correct **combination of subnetworks**.


In [ ]:

def collapse_to_subnetworks(patterns):
    patterns = np.asarray(
        patterns,
        dtype=float,
    )

    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    out = np.zeros(
        (
            patterns.shape[0],
            n_subnetworks,
        )
    )

    for subnet in range(n_subnetworks):
        lo = (
            subnet
            * NODES_PER_SUBNETWORK
        )
        hi = (
            lo
            + NODES_PER_SUBNETWORK
        )

        out[:, subnet] = (
            patterns[:, lo:hi]
            .mean(axis=1)
        )

    return out


TRUE_SUBNET = collapse_to_subnetworks(
    TRUE_MOTIFS
)

AA_SUBNET = collapse_to_subnetworks(
    S_GROUP
)

PCA_SUBNET = collapse_to_subnetworks(
    pca.components_
)

aa_subnet_match = optimal_match(
    TRUE_SUBNET,
    AA_SUBNET,
    absolute=False,
)

pca_subnet_match = optimal_match(
    TRUE_SUBNET,
    PCA_SUBNET,
    absolute=True,
)

subnetwork_recovery_df = pd.DataFrame({
    "method": ["MS-AA", "PCA"],
    "mean_matched_subnetwork_r": [
        np.mean(
            aa_subnet_match["matched"]
        ),
        np.mean(
            pca_subnet_match["matched"]
        ),
    ],
})

display(subnetwork_recovery_df)



## 9. Distinguish “component interpretability” from “subspace capacity”

A reviewer could reasonably object that individual PCA components are not intended to reproduce the generating motifs one-for-one. Therefore we also ask whether the **entire PCA subspace** can reconstruct each true motif.

This is an important control:

- If PCA subspace reconstruction is high but individual component matching is low, PCA has retained the information but **rotated/mixed the motifs**.
- If MS-AA archetypes match the motifs one-to-one, that is precisely the interpretability/disentanglement distinction we want to demonstrate.


In [ ]:

# PCA reconstruction of the true motifs through the fitted PCA subspace
TRUE_MOTIFS_PCA_RECON = pca.inverse_transform(
    pca.transform(
        TRUE_MOTIFS
    )
)

pca_subspace_r = np.array([
    np.corrcoef(
        TRUE_MOTIFS[k],
        TRUE_MOTIFS_PCA_RECON[k],
    )[0, 1]
    for k in range(N_TRUE_MOTIFS)
])

subspace_control_df = pd.DataFrame({
    "true_motif": np.arange(
        N_TRUE_MOTIFS
    ),
    "PCA_subspace_reconstruction_r": (
        pca_subspace_r
    ),
    "best_matched_PCA_component_r": (
        pca_match["matched"]
    ),
    "matched_MSAA_archetype_r": (
        aa_match["matched"]
    ),
})

display(subspace_control_df)

print(
    "\nMean PCA subspace motif reconstruction r:",
    np.mean(pca_subspace_r),
)

print(
    "Mean best one-to-one PCA component r:",
    np.mean(pca_match["matched"]),
)

print(
    "Mean one-to-one MS-AA archetype r:",
    np.mean(aa_match["matched"]),
)



## 10. Optional: inspect the generator matrix `C`

Reviewer 2 specifically asked about the generator matrix. Its exact container structure depends on `MultiSubject_AA.py`, so this section begins by inspecting it rather than assuming an orientation.

The goal is to determine which observations/subjects contribute most strongly to each estimated archetype. Because the simulation knows the true mixture weights, we can eventually ask whether observations contributing strongly to recovered archetype \(k\) are genuinely observations generated near the corresponding true extremal motif.

Run this diagnostic and inspect the printed structure before interpreting `C`.


In [ ]:

C = msaa_fit["C"]

print("Python type:", type(C))

try:
    Carr = np.asarray(C)
    print("np.asarray(C).shape:", Carr.shape)
    print("dtype:", Carr.dtype)
except Exception as e:
    print("Array conversion error:", e)

if isinstance(C, (list, tuple)):
    print("Length:", len(C))

    for i, item in enumerate(C[:3]):
        try:
            arr = np.asarray(
                item,
                dtype=float,
            )
            print(
                f"C[{i}] shape:",
                arr.shape,
                "min/max:",
                np.nanmin(arr),
                np.nanmax(arr),
            )
        except Exception as e:
            print(
                f"C[{i}] could not be summarized:",
                e,
            )



## 11. Negative control: orthogonal Gaussian latent structure

This control deliberately violates the convex-extremal generative assumption. Data are generated from orthogonal Gaussian latent factors.

The expected result is that PCA should be extremely well matched to this geometry. We should **not** expect AA to dominate here.

Because this uses the actual MS-AA optimizer, it may take a little time.


In [ ]:

def simulate_gaussian_control(
    seed=123,
):
    rng = np.random.default_rng(seed)

    V = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    # Orthogonal spatial factors
    Q, _ = np.linalg.qr(
        rng.normal(
            size=(V, K_FIT)
        )
    )

    TRUE_GAUSSIAN_FACTORS = Q.T

    subjects = []

    for s in range(N_SUBJECTS):
        scores = rng.normal(
            size=(
                T_PER_SUBJECT,
                K_FIT,
            )
        )

        X = (
            scores
            @ TRUE_GAUSSIAN_FACTORS
        )

        X += rng.normal(
            0,
            NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)

    subjects_norm, mu, sd = (
        normalize_across_subjects(
            subjects
        )
    )

    transformed_factors = (
        TRUE_GAUSSIAN_FACTORS
        / sd
    )

    return (
        subjects_norm,
        transformed_factors,
    )


GAUSS_SUBJECTS, GAUSS_TRUE = (
    simulate_gaussian_control()
)

gauss_msaa = fit_msaa(
    GAUSS_SUBJECTS,
    K=K_FIT,
    analysis_type="spatial",
    opts=MSAA_OPTS,
    verbose=True,
)

GAUSS_S_GROUP = np.mean(
    np.stack(
        [
            np.asarray(
                sub["S"],
                dtype=float,
            )
            for sub in gauss_msaa[
                "results_subj"
            ]
        ],
        axis=0,
    ),
    axis=0,
)

GAUSS_X = np.vstack(
    GAUSS_SUBJECTS
)

gauss_pca = PCA(
    n_components=K_FIT,
    random_state=RNG_SEED,
).fit(GAUSS_X)

gauss_aa_match = optimal_match(
    GAUSS_TRUE,
    GAUSS_S_GROUP,
    absolute=True,
)

gauss_pca_match = optimal_match(
    GAUSS_TRUE,
    gauss_pca.components_,
    absolute=True,
)

gaussian_control_df = pd.DataFrame({
    "method": ["MS-AA", "PCA"],
    "mean_true_factor_match": [
        np.mean(
            gauss_aa_match["matched"]
        ),
        np.mean(
            gauss_pca_match["matched"]
        ),
    ],
})

display(gaussian_control_df)



## 12. Compact summary figure

For the manuscript, the most useful simulation figure may eventually combine:

1. ground-truth motifs,
2. matched MS-AA archetypes,
3. matched PCA components,
4. one-to-one motif-recovery score,
5. subnetwork-recovery score,
6. reconstruction error.

The point is not that AA must win every metric. The interesting dissociation is:

> **PCA can preserve/reconstruct the information, while AA more directly identifies the latent nonorthogonal extremal motifs that generated it.**


In [ ]:

summary_df = pd.DataFrame({
    "metric": [
        "Observed-data reconstruction MSE",
        "One-to-one motif recovery",
        "Subnetwork motif recovery",
    ],
    "MS-AA": [
        aa_mse,
        np.mean(
            aa_match["matched"]
        ),
        np.mean(
            aa_subnet_match["matched"]
        ),
    ],
    "PCA": [
        pca_mse,
        np.mean(
            pca_match["matched"]
        ),
        np.mean(
            pca_subnet_match["matched"]
        ),
    ],
})

display(summary_df)



# Interpretation guide

### Result that would support the manuscript's proposed distinction

The cleanest outcome is **not** necessarily lower reconstruction error for AA.

A particularly informative result would be:

- PCA reconstructs the data as well as or better than MS-AA;
- the PCA subspace can reconstruct the true motifs reasonably well;
- individual PCA components are mixtures/rotations of several true motifs;
- individual MS-AA archetypes show substantially better one-to-one recovery of the known motifs and their subnetwork composition;
- PCA performs especially well in the orthogonal-Gaussian negative control.

That would give us a precise answer to “why archetypal analysis rather than PCA?”:

> PCA efficiently represents variance within a low-dimensional linear subspace, whereas MS-AA can identify data-anchored extremal patterns when observations are organized as mixtures of nonorthogonal latent motifs.

### Before using this in the manuscript

If the first run looks promising, the next version should add repeated simulations across:

- random seeds,
- noise levels,
- subject heterogeneity,
- motif overlap/nonorthogonality,
- fitted \(K\).

We should also interpret `C` only after confirming its exact orientation from the output diagnostic.
